# Augmentation Strategies — AI Lab Instructor Notebook

*Deep Learning · Hard*



8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Prepare digits train/validation split
*Section: Data + baseline split*

# Augmentation Strategies (NumPy) — Student Lab

We implement image augmentations on `sklearn.datasets.load_digits` (8×8 grayscale) and measure the effect on validation performance.

We’ll compare:
- no augmentation
- label-preserving transforms (shift, noise, cutout)
- label-mixing transforms (mixup, cutmix-style)

## Section 0 — Data + baseline split

FAANG gotcha: augmentation is **train-time only**. Never augment validation/test.

### Task 0.1
Load data, build train/validation split, and normalize with train-only statistics.

In [ ]:
import math
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

digits = load_digits()
X_img = digits.images.astype(float)
y = digits.target.astype(int)

X_tr_img, X_va_img, y_tr, y_va = train_test_split(X_img, y, test_size=0.25, random_state=0, stratify=y)
mu = X_tr_img.mean()
sig = X_tr_img.std() + 1e-12
norm_img = lambda A: (A - mu) / sig
X_tr = norm_img(X_tr_img)
X_va = norm_img(X_va_img)
X_tr_flat = X_tr.reshape(len(X_tr), -1)
X_va_flat = X_va.reshape(len(X_va), -1)
check('shape', X_tr.shape[1:] == (8,8))

In [ ]:
# Checks
check('train_shape', X_tr.shape[1:] == (8, 8))
check('val_shape', X_va.shape[1:] == (8, 8))

## Task 2: Build NumPy MLP utilities
*Section: Model (soft labels)*

## Section 1 — Model: small NumPy MLP supporting soft labels

Mixup/CutMix require *soft targets*. We implement cross-entropy with soft labels.

### Task 1.1
Implement the NumPy MLP helpers and training utilities (`train_model`, soft-label loss, and gradient flow).

In [ ]:
$$10

In [ ]:
# Checks
check('shape', onehot(np.array([0,1]),2).shape == (2,2))

## Task 3: Train a baseline model on flattened pixels.
*Section: Baseline (no augmentation)*

## Section 2 — Baseline (no augmentation)

### Task 2.1
Train a baseline model on flattened pixels.

In [ ]:
p_base = train_model(X_tr_flat, y_tr, X_va_flat, y_va, make_batch=None, lr=0.15, epochs=30, batch_size=128, seed=0)
base_val = accuracy(p_base, X_va_flat, y_va)
print('baseline val', base_val)
check('base_sane', 0.6 <= base_val <= 1.0)

In [ ]:
# Checks
check('base_sane', 0.6 <= base_val <= 1.0)

## Task 4: Implement `random_shift(img, max_shift)`
*Section: Label-preserving augmentations*

## Section 3 — Label-preserving augmentations

We implement augmentations on images (8×8), then flatten.

### Task 3.1
Implement `random_shift(img, max_shift)`

### Task 3.2
Implement `add_noise(img, sigma)`

### Task 3.3
Implement `cutout(img, size)`

In [ ]:
$$11

In [ ]:
# Checks
check('shift_shape', shifted.shape == sample.shape)
check('noise_shape', noisy.shape == sample.shape)
check('cutout_shape', masked.shape == sample.shape)

## Task 5: Train a model with a random composition of (shift + noise + cutout).
*Section: Label-preserving augmentations*

## Section 3 — Label-preserving augmentations

### Task 3.4
Train a model with a random composition of (shift + noise + cutout).

In [ ]:
def make_batch_aug_basic(Xb_flat, yb, K, r):
    B = Xb_flat.shape[0]
    Xb = Xb_flat.reshape(B, 8, 8)
    out = np.empty_like(Xb)
    for i in range(B):
        img = Xb[i]
        img = random_shift(img, max_shift=1, r=r)
        img = add_noise(img, sigma=0.15, r=r)
        img = cutout(img, size=3, r=r)
        out[i] = img
    return out.reshape(B, -1), onehot(yb, K)

p_basic = train_model(X_tr_flat, y_tr, X_va_flat, y_va, make_batch=make_batch_aug_basic, lr=0.15, epochs=30, batch_size=128, seed=1)
basic_val = accuracy(p_basic, X_va_flat, y_va)
print('basic aug val', basic_val)
check('basic_sane', 0.6 <= basic_val <= 1.0)

In [ ]:
# Checks
check('basic_sane', 0.6 <= basic_val <= 1.0)

## Task 6: Implement `mixup_batch(X, y, alpha)` returning mixed inputs and **soft** labels.
*Section: Label-mixing: Mixup*

## Section 4 — Label-mixing: Mixup

### Task 4.1
Implement `mixup_batch(X, y, alpha)` returning mixed inputs and **soft** labels.

In [ ]:
def mixup_batch(Xb_flat, yb, K, r, alpha=0.4):
    B = Xb_flat.shape[0]
    perm = r.permutation(B)
    lam = float(r.beta(alpha, alpha))
    X2 = Xb_flat[perm]
    y1 = onehot(yb, K)
    y2 = onehot(yb[perm], K)
    Xm = lam * Xb_flat + (1.0 - lam) * X2
    ym = lam * y1 + (1.0 - lam) * y2
    return Xm, ym

make_batch_mixup = lambda Xb_flat, yb, K, r: mixup_batch(Xb_flat, yb, K, r, alpha=0.4)
p_mix = train_model(X_tr_flat, y_tr, X_va_flat, y_va, make_batch=make_batch_mixup, lr=0.12, epochs=35, batch_size=128, seed=2)
mix_val = accuracy(p_mix, X_va_flat, y_va)
print('mixup val', mix_val)
check('mix_sane', 0.6 <= mix_val <= 1.0)

In [ ]:
# Checks
check('mix_sane', 0.6 <= mix_val <= 1.0)

## Task 7: Implement `cutmix_batch` returning mixed inputs and soft labels.
*Section: Label-mixing: CutMix-style (patch swap)*

## Section 5 — Label-mixing: CutMix-style (patch swap)

We implement a simplified CutMix on 8×8 by swapping a random rectangle between two images.
The label mixture proportion is the fraction of pixels coming from each image.

### Task 5.1
Implement `cutmix_batch` returning mixed inputs and soft labels.

In [ ]:
$f

In [ ]:
# Checks
check('cutmix_sane', 0.6 <= cmix_val <= 1.0)

## Task 8: Compare baseline vs augmentations
*Section: Compare*

## Section 6 — Compare

Interpret the results. You should not expect augmentation to always win on digits; the goal is to reason about *why* it helped or hurt.

In [ ]:
res = {'baseline': base_val, 'basic_aug': basic_val, 'mixup': mix_val, 'cutmix': cmix_val}
print(res)
best = max(res, key=res.get)
print('best', best, res[best])
check('best_sane', 0.6 <= res[best] <= 1.0)

In [ ]:
# Checks
check('best_sane', 0.6 <= res[best] <= 1.0)